## Check which unmatched records remain in the grid panel

A record can fail to match a township but still be assigned to a 5 km grid cell that overlaps Jiangxi. This first check compares the unmatched records with `operation_with_exact_times.csv` to show which records remain in the analysis panel.

In [3]:
import pandas as pd

data_dir = "/Users/oliviakuang/Documents/GitHub/Cloudseeding2"

unmatched = pd.read_csv(
    f"{data_dir}/check/unmatched_operations.csv"
)

retained = pd.read_csv(
    f"{data_dir}/intermediate/operation_with_exact_times.csv"
)

# Correct the known swapped coordinates
swap = (
    unmatched["date"].eq("2022-10-27")
    & unmatched["lon"].eq(29.043)
    & unmatched["lat"].eq(115.560)
    & unmatched["city_o"].eq("九江市")
)

unmatched.loc[swap, ["lon", "lat"]] = [115.560, 29.043]

keys = ["date", "lon", "lat", "city_o", "county_o"]

review = unmatched.merge(
    retained[keys + ["cell_id"]].drop_duplicates(),
    on=keys,
    how="left",
)

review["retained_in_grid"] = review["cell_id"].notna()

print(
    review[
        keys + ["cell_id", "retained_in_grid"]
    ].to_string(index=False)
)

summary = (
    review.groupby(
        ["lon", "lat", "city_o", "county_o", "cell_id", "retained_in_grid"],
        dropna=False,
    )
    .size()
    .reset_index(name="number_of_records")
)

print(summary.to_string(index=False))

      date       lon      lat city_o county_o cell_id  retained_in_grid
2021-09-06 115.38100 24.56500    赣州市      寻乌县     NaN             False
2022-10-27 115.56000 29.04300    九江市      永修县  100_39              True
2023-07-31 115.76900 24.81400    赣州市      寻乌县    6_42              True
2023-07-30 115.76900 24.81400    赣州市      寻乌县    6_42              True
2023-07-18 115.76900 24.81400    赣州市      寻乌县    6_42              True
2023-07-18 115.76900 24.81400    赣州市      寻乌县    6_42              True
2023-07-17 115.76900 24.81400    赣州市      寻乌县    6_42              True
2023-07-16 115.76900 24.81400    赣州市      寻乌县    6_42              True
2023-03-11 118.64000 28.49000    上饶市      广丰区     NaN             False
2023-03-11 118.64000 28.49000    上饶市      广丰区     NaN             False
2024-12-07 115.76900 24.81400    赣州市      寻乌县    6_42              True
2024-11-14 115.76900 24.81400    赣州市      寻乌县    6_42              True
2024-11-13 115.76900 24.81400    赣州市      寻乌县    6_42           

### What this check shows

Of the 31 unmatched records, 26 are kept in the grid panel because their calculated 5 km cells overlap Jiangxi. These are the corrected longitude–latitude swap in cell `100_39`, the 24 repeated 寻乌 records in cell `6_42`, and the 浮梁 record in cell `113_80`. 

The other five records are excluded because their calculated grid cells do not overlap Jiangxi: one 寻乌 record at `(115.381, 24.565)`, two 广丰 records at `(118.640, 28.490)`, and two 广昌 records at `(116.34833, 25.84472)` and `(116.34833, 25.66111)`.

At this stage, the longitude–latitude swap is the only error that can be corrected confidently. The remaining records need more evidence from the original data or another location source before making manual changes.

## Inspect distances from township and Jiangxi boundaries

This section checks the 7 unique coordinate locations against the xiangzhen.shp and jiangxi_shape.shp, which are the same shapefiles used in the pipeline. Distances are measured in meters after projecting the points and polygons to EPSG:32650. The calculations distinguish points that are inside a polygon, exactly on a boundary, just outside a boundary, or well outside the available township coverage.

In [4]:
import geopandas as gpd

townships = gpd.read_file(
    f"{data_dir}/township_shapefile/xiangzhen.shp"
)
townships = townships.loc[
    townships["省"].eq("江西省"),
    ["市", "县", "乡", "geometry"],
].copy()
townships = townships.to_crs("EPSG:4326")

jiangxi = gpd.read_file(
    f"{data_dir}/jiangxi_shapefile/jiangxi_shape.shp"
).to_crs("EPSG:4326")

# Repeated coordinates receive the same geometric diagnosis.
coordinate_groups = (
    pd.read_csv(f"{data_dir}/check/unmatched_operations.csv")
    .groupby(["lon", "lat", "city_o", "county_o"], dropna=False)
    .agg(
        record_count=("date", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reset_index()
)

points = gpd.GeoDataFrame(
    coordinate_groups,
    geometry=gpd.points_from_xy(
        coordinate_groups["lon"], coordinate_groups["lat"]
    ),
    crs="EPSG:4326",
)

In [9]:
def normalize_county_name(value):
    name = str(value).strip()
    for suffix in ("县", "区", "市"):
        if name.endswith(suffix):
            return name[:-len(suffix)]
    return name

townships_m = townships.to_crs("EPSG:32650")
jiangxi_union = jiangxi.geometry.union_all()
jiangxi_union_m = jiangxi.to_crs("EPSG:32650").geometry.union_all()
township_county_norm = townships["县"].map(normalize_county_name)

boundary_results = []

for _, row in points.iterrows():
    valid_lon_lat = -180 <= row["lon"] <= 180 and -90 <= row["lat"] <= 90
    result = {
        "lon": row["lon"],
        "lat": row["lat"],
        "record_count": row["record_count"],
        "city_o": row["city_o"],
        "county_o": row["county_o"],
        "first_date": row["first_date"],
        "last_date": row["last_date"],
        "valid_lon_lat": valid_lon_lat,
    }

    if not valid_lon_lat:
        boundary_results.append({
            **result,
            "inside_or_on_jiangxi": False,
            "jiangxi_distance_m": pd.NA,
            "inside_or_on_any_township": False,
            "nearest_township_distance_m": pd.NA,
            "nearest_city": pd.NA,
            "nearest_county": pd.NA,
            "nearest_town": pd.NA,
            "reported_county_distance_m": pd.NA,
            "interpretation": "Invalid longitude/latitude range; inspect for a coordinate swap.",
        })
        continue

    point = row.geometry
    point_m = gpd.GeoSeries(
        [point], crs="EPSG:4326"
    ).to_crs("EPSG:32650").iloc[0]
    distances = townships_m.geometry.distance(point_m)
    nearest_index = distances.idxmin()
    nearest = townships.loc[nearest_index]
    nearest_distance = float(distances.loc[nearest_index])

    reported_county = normalize_county_name(row["county_o"])
    same_county = townships.index[township_county_norm.eq(reported_county)]
    reported_county_distance = (
        float(distances.loc[same_county].min())
        if len(same_county)
        else pd.NA
    )

    if nearest_distance == 0:
        interpretation = "On a boundary or covered by a township polygon."
    elif nearest_distance <= 100:
        interpretation = "Just outside; rounding or boundary-data differences are possible."
    elif nearest_distance <= 5000:
        interpretation = "Near the boundary, but the original location should be checked."
    else:
        interpretation = "Well outside township coverage; inspect the coordinate and reported location."

    boundary_results.append({
        **result,
        "inside_or_on_jiangxi": bool(jiangxi_union.covers(point)),
        "jiangxi_distance_m": float(jiangxi_union_m.distance(point_m)),
        "inside_or_on_any_township": bool(townships.geometry.covers(point).any()),
        "nearest_township_distance_m": nearest_distance,
        "nearest_city": nearest["市"],
        "nearest_county": nearest["县"],
        "nearest_town": nearest["乡"],
        "reported_county_distance_m": reported_county_distance,
        "interpretation": interpretation,
    })

boundary_review = (
    pd.DataFrame(boundary_results)
    .sort_values(["record_count", "lon"], ascending=[False, True])
    .reset_index(drop=True)
)
boundary_review

,lon,lat,record_count,city_o,county_o,first_date,last_date,valid_lon_lat,inside_or_on_jiangxi,jiangxi_distance_m,inside_or_on_any_township,nearest_township_distance_m,nearest_city,nearest_county,nearest_town,reported_county_distance_m,interpretation
0,115.76900,24.81400,24,赣州市,寻乌县,2023-07-16,2025-09-20,True,False,83.50355,False,83.50355,赣州市,寻乌县,南桥镇,83.50355,Just outside; rounding or boundary-data differ...
1,118.64000,28.49000,2,上饶市,广丰区,2023-03-11,2023-03-11,True,False,16033.693111,False,16033.693111,上饶市,广丰县,东阳乡,16033.693111,Well outside township coverage; inspect the co...
2,29.04300,115.56000,1,九江市,永修县,2022-10-27,2022-10-27,False,False,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,Invalid longitude/latitude range; inspect for ...
3,115.38100,24.56500,1,赣州市,寻乌县,2021-09-06,2021-09-06,True,False,19340.394158,False,19340.394158,赣州市,定南县,鹅公镇,19733.826156,Well outside township coverage; inspect the co...
4,116.34833,25.66111,1,抚州市,广昌,2020-08-12,2020-08-12,True,False,20493.382058,False,20493.382058,赣州市,瑞金市,泽覃乡,93247.152588,Well outside township coverage; inspect the co...
5,116.34833,25.84472,1,抚州市,广昌,2020-07-29,2020-07-29,True,False,10249.02575,False,10249.02575,赣州市,瑞金市,日东乡,72946.841195,Well outside township coverage; inspect the co...
6,117.67694,29.62972,1,景德镇,浮梁,2020-12-26,2020-12-26,True,False,2609.06598,False,2609.06598,景德镇市,浮梁县,瑶里镇,2609.06598,"Near the boundary, but the original location s..."


In [6]:
# Verify the known longitude/latitude correction separately.
corrected_point = gpd.GeoSeries(
    gpd.points_from_xy([115.560], [29.043]),
    crs="EPSG:4326",
).iloc[0]

corrected_match = townships.loc[
    townships.geometry.covers(corrected_point),
    ["市", "县", "乡"],
]

print("Corrected point lies inside or on Jiangxi:", jiangxi_union.covers(corrected_point))
print("Township assignment after correction:")
display(corrected_match)

Corrected point lies inside or on Jiangxi: True
Township assignment after correction:


,市,县,乡
43461,九江市,永修县,云山企业集团


### Notes on the result columns

Each row represents one coordinate and reported city-county combination. Repeated records at the same location are grouped together.

**Record information**

- `lon`, `lat`: coordinates from the operation data.
- `record_count`: number of records sharing the same coordinate and reported location.
- `city_o`, `county_o`: city and county reported in the original file.
- `first_date`, `last_date`: date range covered by the grouped records.
- `valid_lon_lat`: checks only whether longitude is between -180 and 180 and latitude is between -90 and 90. It does not tell us whether the coordinate is correct.

**Jiangxi boundary**

- `inside_or_on_jiangxi`: whether the point is inside Jiangxi or exactly on its boundary.
- `jiangxi_distance_m`: shortest distance to the Jiangxi polygon, measured in meters after projection to EPSG:32650. It is 0 for points inside or on the boundary and positive for points outside.

**Township boundary**

- `covered_by_any_township`: whether the point is inside a township or exactly on its boundary.

**Nearest locations and distances**

- `nearest_township_distance_m`: distance to the closest township polygon. A value of 0 means the point is inside or on one.
- `nearest_city`, `nearest_county`, `nearest_town`: labels from that closest polygon. They describe what is nearest; they do not prove where the operation occurred.
- `reported_county_distance_m`: distance to the closest township polygon in the county reported by the source data. County suffixes such as 县, 区, and 市 are removed before matching names.

### What I found

The 31 records reduce to 7 unique coordinate locations. None of the unmatched points lies inside or on the boundary of any township polygon, and none lies inside or exactly on the Jiangxi boundary. There `inside_or_on_jiangxi`and `inside_or_on_any_township` values are all `False`.

| Records | Reported location | Coordinates | Observation | Current decision |
|---:|---|---|---|---|
| 1 | 九江市–永修县 | `(29.043, 115.560)` | The latitude is outside the possible range. Swapping the values to `(115.560, 29.043)` places the point inside 永修县–云山企业集团. | Retain after the manual coordinate swap |
| 24 | 赣州市–寻乌县 | `(115.769, 24.814)` | The shared coordinate is about 84 m outside Jiangxi and the nearest township polygon, 寻乌县–南桥镇. Because all 24 records use exactly the same point, it may be a rounded or standardized site coordinate. | Retain under the existing grid rule |
| 1 | 赣州市–寻乌县 | `(115.381, 24.565)` | The point is about 19.3 km outside Jiangxi. Its nearest township is 定南县–鹅公镇, and it is about 19.7 km from the reported 寻乌县 polygons. | Drop under the existing grid rule |
| 2 | 上饶市–广丰区 | `(118.640, 28.490)` | The point is about 16.0 km outside Jiangxi. The nearest township is 广丰县–东阳乡, so the coordinate is in the general area of the reported county but not close enough to treat as a simple boundary-rounding issue. | Dropped under the existing grid rule|
| 1 | 抚州市–广昌 | `(116.34833, 25.84472)` | The point is about 10.2 km outside Jiangxi and about 72.9 km from the reported 广昌县 polygons. The nearest township is 瑞金市–日东乡. | Drop under the existing grid rule; reported location and coordinate conflict. |
| 1 | 抚州市–广昌 | `(116.34833, 25.66111)` | The point is about 20.5 km outside Jiangxi and about 93.2 km from the reported 广昌县 polygons. The nearest township is 瑞金市–泽覃乡. | Drop under the existing grid rule; reported location and coordinate conflict. |
| 1 | 景德镇–浮梁 | `(117.67694, 29.62972)` | The point is about 2.6 km outside Jiangxi and the nearest township, 浮梁县–瑶里镇. This is relatively close, but not an exact-boundary case. | Retain under the existing grid rule|



#### Retained records

Of the 25 unmatched records retained without coordinate corrections in the pipeline:

- **24 寻乌 records** share the coordinate `(115.769, 24.814)`, which is only about **84 m outside** Jiangxi and the nearest township polygon, 寻乌县–南桥镇. 

- **1 浮梁 record** is about **2.6 km outside** Jiangxi and the nearest township polygon, 浮梁县–瑶里镇. 

Possible explanations for these unmatching include:

- The coordinate represents a standardized operating site instead of the exact operation point (particularly likely for the 24 寻乌 records because they repeatedly use the same coordinate)
- The operation occurred just outside the Jiangxi boundary but remained administratively associated with the county in Jiangxi (likely given the short distances)
- The boundary differs across shapefile sources or dates
- The coordinate or reported location have a source-data error (not so likely)

The original operation record and an independent location source are needed to distinguish among these explanations. Regardless, based on the relatively short distances, these records seem more geographically plausible than the excluded cases, so keeping them under the existing grid-cell rule seems to be appropriate.

#### Corrected coordinate swap

The longitude–latitude swap is the only record with a well supported correction. After changing `(29.043, 115.560)` to `(115.560, 29.043)`, the point falls inside 永修县–云山企业集团. Including this corrected record brings the total number retained to 26.

#### Excluded records

The remaining 5 records are approximately **10–20 km outside Jiangxi**, and their calculated grid cells do not overlap the province. Their exclusion under the grid-cell rule is therefore consistent with the boundary-distance results. In particular, ...

The 1 isolated 寻乌 record is about **19.3 km outside Jiangxi**. Its nearest county is **定南县**, rather than the reported **寻乌县**, and it is about 19.7 km from the 寻乌 township polygon. This is a great inconsistency between the recorded coordinate and reported county. The original file comparison below further suggests that there may be a coordinate entry error.

The 2 广昌 coordinates are **~73–93 km from the nearest township polygons in the reported 广昌县**. Based on their coordinates, their nearest township polygons are instead in **赣州市–瑞金市**, but not **抚州市–广昌县**. These large discrepancies are unlikely to result from coordinate rounding or boundary differences between  datasets. The comparison with the original file below provides further evidence for a latitude entry error.


## Additional inpections

### Comparison with the original operation csvs

Looking back at the original 2020–2025 operation files gives us more information:

- **The 永修 record** (kept after manual coordinate swap): The original 2022 row reports a rocket operation in 九江市–永修县 but records latitude as `115.560`, which is outside the possible latitude range. Swapping the two values places the point inside 永修县, so this manual correction can be made confidently from the record itself.

- **The 24 repeated 寻乌 records** (kept): All 24 operations from 2023–2025 report 赣州市–寻乌县, use a smoke furnace (烟炉), and have the same coordinate and elevation of 601 m. Their dates, times, and operation quantities vary, so they are separate operations instead of duplicated rows. This strongly suggests a that these operations happen at fixed smoke-furnace site or are documented with a standardized site coordinate. The coordinate is recorded to only three decimal places, so coordinate rounding imprecision is a plausible explanation for the 84 m boundary gap.

- **The 浮梁 record** (kept): The original 2020 row identifies the site as `瑶里镇虎头岗360222003` and the equipment as an iodide-silver ground generator ("碘化银地面发生器"). This supports the record's association with 浮梁县–瑶里镇 and suggests that the coordinate may represent a fixed site. The 2.6 km distance may indicate (1)that the site is very close or just across the mapped boundary, (2)a difference in boundary data, (3)an imprecise site coordinate. Given its relative proximity and the fact that its assigned grid cell overlaps Jiangxi, retaining this record under the existing grid-cell rule seems appropriate

- **The 2 广丰 records** (dropped): These are two rocket operations on March 11, 2023, about one hour apart. They have the same coordinate, elevation of 100 m, equipment type, quantity, area, effect, and service field. They seem to be 2 consecutive operations from the same site. However, the file has no site name or remarks to explain why the shared point is about 16 km outside Jiangxi.

- **The isolated 2021 寻乌 record** (dropped): This is a single rocket operation at `(115.381, 24.565)`, with an elevation of 298 m. The 2021 file contains 20+ 寻乌 operations at 8 coordinate locations, but this coordinate shows up only once and is much farther south and west than the other 寻乌 points. I guess the coordinate has a documentation or data entry error, but this cannot be confirmed from the current file.

- **The 2 广昌 records** (dropped): Both report the detailed site `赤水镇回辛村361030005`. Two other 2020 operations at the same reported site use nearly identical longitudes and latitudes beginning with `26`, and those points fall inside 广昌县. The two unmatched rows have approximately the same longitude (identical to the 2nd decimal point) but have latitudes beginning with `25`. Changing only the leading latitude degree from `25` to `26` would place both points inside 广昌县. This is a strong evidence of a latitude entry error, but we don't have access to the exact intended coordinates, so we cannot manually correct the values at this point.

### Online searching

Jiangxi regulations require local meteorological authorities to announce weather operations in advance, including the operation period, area, and equipment type.(`check/江西省人工影响天气管理条例`) However, I could not find public notices corresponding to these particular unmatched records or any publicly available source linking their exact site codes to coordinates. 

The notices that are searchable online tend to be broad seasonal announcements covering an entire administrative area rather than individual operations or fixed sites.(`https://www.sohu.com/a/971502594_121106832`) The relevant notices may have been posted locally or removed after the operation period. 

## Conclusion

The inspection generally supports the current grid assignment based on coordinate and the grid cell filter. The retained records are relatively close to Jiangxi and belong to grid cells that overlap the province, while the excluded records generally have greater geographic gaps.

The isolated 寻乌 record and the two 广昌 records are the most suspicious for a data entry error-- based on their original files and and the inconsistencies between their recorded coordinates and reported locations. However, the available evidence does not help us identify the correct coordinates. They should thus remain excluded and cannot be manually cleaned until further review.